# Cryptocurrency Forecasting Pipeline (BTC/ETH)
Mount Google Drive, lưu CSV theo thư mục **Raw / Processed / Results**, và in kết quả theo từng timeframe (horizon).

In [24]:
# @title # Mount Data
from google.colab import drive
import os

# Ensure the mount point is empty before mounting
if os.path.exists('/content/drive'):
    # Only remove if it's a directory and not already a mount point
    if os.path.isdir('/content/drive') and not os.path.ismount('/content/drive'):
        print("Clearing existing /content/drive directory...")
        for root, dirs, files in os.walk('/content/drive', topdown=False):
            for name in files:
                os.remove(os.path.join(root, name))
            for name in dirs:
                os.rmdir(os.path.join(root, name))
    elif os.path.isfile('/content/drive'):
        print("Removing file at /content/drive...")
        os.remove('/content/drive')

# Attempt to create the directory if it doesn't exist (it should be empty now)
os.makedirs('/content/drive', exist_ok=True)

drive.mount('/content/drive', force_remount=True)

BASE_DIR = "/content/drive/MyDrive/Crypto Research"
RAW_DIR = os.path.join(BASE_DIR, "DATA", "Raw")
PROC_DIR = os.path.join(BASE_DIR, "DATA", "Processed")
RES_DIR = os.path.join(BASE_DIR, "RESULTS")

for d in [RAW_DIR, PROC_DIR, RES_DIR]:
    os.makedirs(d, exist_ok=True)

print("Raw dir:", RAW_DIR)
print("Processed dir:", PROC_DIR)
print("Results dir:", RES_DIR)

Mounted at /content/drive
Raw dir: /content/drive/MyDrive/Crypto Research/DATA/Raw
Processed dir: /content/drive/MyDrive/Crypto Research/DATA/Processed
Results dir: /content/drive/MyDrive/Crypto Research/RESULTS


In [35]:
import os
import io
import re
import json
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import List

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Bidirectional, LSTM

BITINFOCHARTS_BASE = "https://bitinfocharts.com/comparison/"

# Valid BitInfoCharts features (from repo generate_features.py)
BITINFO_FEATURES = [
    "transactions",
    "size",
    "sentbyaddress",
    "transactionfees",
    "blocktime",
    "difficulty",
    "hashrate",
    "transactionvalue",
    "mediantransactionvalue",
    "profitability",
    "activeaddresses",
    "sentinusd",
    "top100cap",
    "fee-to-reward-ratio",
    "mediantransactionfee",
    "price",
]

# Optional features (try; skip if 404)
OPTIONAL_FEATURES = [
    "sentcoin",
]

# Rename to spec-friendly column names
RENAME_MAP = {
    "size": "blocksize",
    "transactionfees": "average_tx_fee",
    "mediantransactionfee": "median_tx_fee",
    "transactionvalue": "average_tx_value",
    "mediantransactionvalue": "median_tx_value",
    "profitability": "mining_profitability",
    "fee-to-reward-ratio": "fee_to_reward",
}

WINDOWS = [3, 7, 14, 30, 90]
HORIZONS = [1, 7, 14, 30, 60, 90]

INTERVALS = {
    "interval1": ("2013-04-01", "2016-04-01"),
    "interval2": ("2013-04-01", "2017-04-01"),
    "interval3": ("2013-04-01", "2019-12-31"),
}
CUSUM_TEST = ("2020-01-01", "2022-01-01")

def save_csv(df, dir_path, name):
    path = os.path.join(dir_path, name)
    df.to_csv(path, index=False)
    print(f"Saved: {path}")
    return path

def load_csv(dir_path, name):
    path = os.path.join(dir_path, name)
    df = pd.read_csv(path)
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])
    print(f"Loaded: {path}")
    return df

def load_state(path):
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    return {"completed": []}

def save_state(state, path):
    with open(path, "w") as f:
        json.dump(state, f, indent=2)

def is_completed(state, coin, interval_name, horizon):
    key = f"{coin}|{interval_name}|{horizon}"
    return key in state.get("completed", [])

def mark_completed(state, coin, interval_name, horizon):
    key = f"{coin}|{interval_name}|{horizon}"
    if key not in state.get("completed", []):
        state.setdefault("completed", []).append(key)

def update_best(result_dir, coin_tag, interval_name, horizon, metrics_row):
    # metrics_row: [interval, horizon, rmse, mae, mape, r2]
    best_path = os.path.join(result_dir, f"{coin_tag}_best_tf{horizon}.csv")
    new_rmse = metrics_row[2]
    if os.path.exists(best_path):
        best_df = pd.read_csv(best_path)
        best_rmse = best_df.iloc[0]["RMSE"]
        if new_rmse >= best_rmse:
            return
    best_df = pd.DataFrame(
        {
            "interval": metrics_row[0],
            "horizon_days": metrics_row[1],
            "RMSE": metrics_row[2],
            "MAE": metrics_row[3],
            "MAPE": metrics_row[4],
            "R2": metrics_row[5],
        }
    )
    save_csv(best_df, result_dir, f"{coin_tag}_best_tf{horizon}.csv")

def plot_predictions(dates, y_true, y_pred, out_path, title):
    plt.figure(figsize=(12, 6))
    plt.plot(dates, y_true, label="Actual", linewidth=1.5)
    plt.plot(dates, y_pred, label="Predicted", linewidth=1.5)
    plt.title(title)
    plt.xlabel("Date")
    plt.ylabel("Price")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    print(f"Chart saved to: {out_path}") # Confirmation print
    plt.close()


In [26]:
def fetch_bitinfocharts_feature(feature: str, coin: str, session: requests.Session) -> pd.DataFrame:
    url = f"https://bitinfocharts.com/comparison/{coin}-{feature}.html#alltime"
    resp = session.get(url, headers={"User-Agent": "Mozilla/5.0"})
    if resp.status_code == 404:
        print(f"Warning: {feature} not found for {coin} (404)")
        return pd.DataFrame()
    resp.raise_for_status()
    pattern = r'\[new Date\("(.*?)"\),(.*?)\]'
    matches = re.findall(pattern, resp.text)
    if not matches:
        print(f"Warning: no data parsed for {feature}")
        return pd.DataFrame()
    dates = [pd.to_datetime(m[0]) for m in matches]
    values = [float(m[1]) if m[1] != "null" else np.nan for m in matches]
    df = pd.DataFrame({"date": dates, feature: values})
    return df

def fetch_all_bitinfocharts(coin: str) -> pd.DataFrame:
    df_all = pd.DataFrame()
    with requests.Session() as session:
        for feature in BITINFO_FEATURES + OPTIONAL_FEATURES:
            df_feat = fetch_bitinfocharts_feature(feature, coin, session)
            if df_feat.empty:
                continue
            df_all = df_feat if df_all.empty else df_all.merge(df_feat, on="date", how="outer")

    if df_all.empty:
        raise RuntimeError("No BitInfoCharts data fetched.")

    # Rename columns to spec-friendly names
    df_all = df_all.rename(columns=RENAME_MAP)

    # Build OHLC from daily price (BitInfoCharts provides daily price)
    if "price" in df_all.columns:
        df_all["close"] = df_all["price"]
        df_all["open"] = df_all["price"]
        df_all["high"] = df_all["price"]
        df_all["low"] = df_all["price"]
        df_all = df_all.drop(columns=["price"])

    if "close" in df_all.columns:
        for col in ["open", "high", "low"]:
            if col not in df_all.columns:
                df_all[col] = df_all["close"]

    df_all = df_all.sort_values("date")
    return df_all

In [27]:
def SMA(series, window): return series.rolling(window).mean()
def EMA(series, window): return series.ewm(span=window, adjust=False).mean()
def WMA(series, window):
    weights = np.arange(1, window + 1)
    return series.rolling(window).apply(lambda x: np.dot(x, weights) / weights.sum(), raw=True)
def RSI(series, window=14):
    delta = series.diff()
    gain = np.where(delta > 0, delta, 0)
    loss = np.where(delta < 0, -delta, 0)
    avg_gain = pd.Series(gain).rolling(window).mean()
    avg_loss = pd.Series(loss).rolling(window).mean()
    rs = avg_gain / (avg_loss + 1e-9)
    return 100 - (100 / (1 + rs))
def STD(series, window): return series.rolling(window).std()
def VAR(series, window): return series.rolling(window).var()
def TRIX(series, window):
    ema1 = series.ewm(span=window, adjust=False).mean()
    ema2 = ema1.ewm(span=window, adjust=False).mean()
    ema3 = ema2.ewm(span=window, adjust=False).mean()
    return ema3.pct_change() * 100
def ROC(series, window): return series.pct_change(periods=window) * 100

def add_indicators(df: pd.DataFrame, base_cols: List[str]) -> pd.DataFrame:
    for col in base_cols:
        for w in WINDOWS:
            df[f"{col}_SMA_{w}"] = SMA(df[col], w)
            df[f"{col}_EMA_{w}"] = EMA(df[col], w)
            df[f"{col}_RSI_{w}"] = RSI(df[col], w)
            df[f"{col}_WMA_{w}"] = WMA(df[col], w)
            df[f"{col}_STD_{w}"] = STD(df[col], w)
            df[f"{col}_VAR_{w}"] = VAR(df[col], w)
            df[f"{col}_TRIX_{w}"] = TRIX(df[col], w)
            df[f"{col}_ROC_{w}"] = ROC(df[col], w)
    return df

In [28]:
def preprocess_split(df, target_col):
    X = df.drop(columns=[target_col])
    y = df[target_col].values

    imputer = SimpleImputer(strategy="most_frequent")
    X_imputed = imputer.fit_transform(X)

    X_train, X_test, y_train, y_test = train_test_split(
        X_imputed, y, test_size=0.33, random_state=42, shuffle=False
    )

    scaler = MinMaxScaler(feature_range=(0, 1))
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    return X_train_scaled, X_test_scaled, y_train, y_test, X.columns

def select_features_mdi(X_train, y_train, feature_names):
    rf = RandomForestRegressor(bootstrap=False, random_state=42)
    rf.fit(X_train, y_train)

    result = permutation_importance(rf, X_train, y_train, random_state=42)
    mean = result.importances_mean
    std = result.importances_std

    keep_idx = []
    for i in range(len(feature_names)):
        if mean[i] - (2 * std[i]) > 0:
            keep_idx.append(i)
    return keep_idx

In [29]:
def build_bilstm(input_shape):
    model = Sequential()
    model.add(Bidirectional(LSTM(128, activation="relu"), input_shape=input_shape))
    model.add(Dropout(0.2))
    model.add(Dense(1))
    model.compile(loss="mse", optimizer="adam")
    return model

def make_sliding_window(X, y, window=1):
    Xs, ys = [], []
    for i in range(len(X) - window):
        Xs.append(X[i:i+window])
        ys.append(y[i+window])
    return np.array(Xs), np.array(ys)

def CUSUM_Control_Chart(predictions, actuals, model, X_recent, y_recent):
    preds = np.array(predictions)
    n = len(preds)
    variance = np.var(preds)
    std = np.sqrt(variance)
    std = std / n

    upper = std * 3
    lower = -std * 3

    cumsum = 0
    for i in range(len(preds)):
        deviation = preds[i] - actuals[i]
        cumsum += deviation
        if cumsum > upper or cumsum < lower:
            print("CUSUM WARNING: bias detected, retraining on recent data...")
            model.fit(X_recent, y_recent, epochs=10, batch_size=32, verbose=0)
            break

def report_metrics(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mae = np.mean(np.abs(y_true - y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100
    r2 = r2_score(y_true, y_pred)
    return rmse, mae, mape, r2

In [30]:
def run_pipeline(coin: str, coin_id: str):
    coin_tag = "bitcoin" if coin == "btc" else coin

    state_dir = os.path.join(RES_DIR, "State")
    os.makedirs(state_dir, exist_ok=True)
    state_path = os.path.join(state_dir, f"{coin_tag}_run_state.json")
    state = load_state(state_path)

    # Step 1: Fetch + Save raw
    df_metrics = fetch_all_bitinfocharts(coin)
    df_raw = df_metrics.sort_values("date")
    save_csv(df_raw, RAW_DIR, f"{coin_tag}_raw.csv")

    # Step 2: Load raw + Feature engineering
    df_raw = load_csv(RAW_DIR, f"{coin_tag}_raw.csv")
    if "close" not in df_raw.columns:
        raise RuntimeError("Missing close price in BitInfoCharts data.")
    base_cols = [c for c in df_raw.columns if c != "date"]
    df_feat = add_indicators(df_raw, base_cols)
    save_csv(df_feat, PROC_DIR, f"{coin_tag}_features.csv")

    # Step 3: Load features + intervals + model
    df_feat = load_csv(PROC_DIR, f"{coin_tag}_features.csv")

    interval_dir = os.path.join(PROC_DIR, "Intervals", coin_tag)
    pred_dir = os.path.join(RES_DIR, "Predictions", coin_tag)
    result_dir = os.path.join(RES_DIR, "Tables", coin_tag)
    charts_dir = os.path.join(RES_DIR, "Charts", coin_tag)
    for d in [interval_dir, pred_dir, result_dir, charts_dir]:
        os.makedirs(d, exist_ok=True)

    # Load existing results if available (resume)
    results_map = {}
    existing_path = os.path.join(result_dir, f"{coin_tag}_results_all.csv")
    if os.path.exists(existing_path):
        existing_df = pd.read_csv(existing_path)
        for _, r in existing_df.iterrows():
            key = (r["interval"], int(r["horizon_days"]))
            results_map[key] = [r["interval"], int(r["horizon_days"]), r["RMSE"], r["MAE"], r["MAPE"], r["R2"]]

    for interval_name, (start, end) in INTERVALS.items():
        df_interval = df_feat[(df_feat["date"] >= start) & (df_feat["date"] <= end)].copy()
        save_csv(df_interval, interval_dir, f"{coin_tag}_{interval_name}.csv")

        for horizon in HORIZONS:
            if is_completed(state, coin_tag, interval_name, horizon):
                print(f"Skip {coin_tag} | {interval_name} | TF {horizon}d (resume)")
                continue

            target = f"close_t+{horizon}"
            df_interval[target] = df_interval["close"].shift(-horizon)
            data = df_interval.dropna().copy()
            date_series = data["date"].reset_index(drop=True)

            # Save horizon dataset
            save_csv(data, interval_dir, f"{coin_tag}_{interval_name}_h{horizon}_dataset.csv")

            X_train, X_test, y_train, y_test, feature_names = preprocess_split(data.drop(columns=["date"]), target)
            keep_idx = select_features_mdi(X_train, y_train, feature_names)
            X_train_sel = X_train[:, keep_idx]
            X_test_sel = X_test[:, keep_idx]

            X_train_3d, y_train_3d = make_sliding_window(X_train_sel, y_train, window=1)
            X_test_3d, y_test_3d = make_sliding_window(X_test_sel, y_test, window=1)

            # align dates with y_test_3d
            split_idx = len(date_series) - len(y_test)
            test_dates = date_series.iloc[split_idx + 1:].reset_index(drop=True)

            model = build_bilstm((X_train_3d.shape[1], X_train_3d.shape[2]))
            model.fit(X_train_3d, y_train_3d, epochs=10, batch_size=32, verbose=0)
            preds = model.predict(X_test_3d).flatten()

            # Save predictions CSV (with date)
            pred_df = pd.DataFrame({"date": test_dates, "y_true": y_test_3d, "y_pred": preds})
            save_csv(pred_df, pred_dir, f"{coin_tag}_{interval_name}_h{horizon}_predictions.csv")

            # Plot prediction line chart
            chart_path = os.path.join(charts_dir, f"{coin_tag}_{interval_name}_h{horizon}_chart.png")
            plot_predictions(pred_df["date"], pred_df["y_true"], pred_df["y_pred"], chart_path,
                             f"{coin_tag.upper()} {interval_name} TF {horizon}d Prediction")

            rmse, mae, mape, r2 = report_metrics(y_test_3d, preds)
            row = [interval_name, horizon, rmse, mae, mape, r2]
            results_map[(interval_name, horizon)] = row
            update_best(result_dir, coin_tag, interval_name, horizon, row)

            print(f"{coin_tag} | {interval_name} | TF {horizon}d -> RMSE={rmse:.4f}, MAE={mae:.4f}, MAPE={mape:.2f}%, R2={r2:.4f}")

            mark_completed(state, coin_tag, interval_name, horizon)
            save_state(state, state_path)

    results_df = pd.DataFrame(list(results_map.values()), columns=["interval", "horizon_days", "RMSE", "MAE", "MAPE", "R2"])
    results_df = results_df.sort_values(["interval", "horizon_days"]).reset_index(drop=True)
    save_csv(results_df, result_dir, f"{coin_tag}_results_all.csv")

    # Save results per timeframe (tf)
    for horizon in HORIZONS:
        tf_df = results_df[results_df["horizon_days"] == horizon].copy()
        save_csv(tf_df, result_dir, f"{coin_tag}_results_tf{horizon}.csv")

    print("\nSummary results (all):")
    print(results_df.head())

    # Step 4: CUSUM out-of-sample
    start, end = CUSUM_TEST
    df_cusum = df_feat[(df_feat["date"] >= start) & (df_feat["date"] <= end)].copy()
    df_cusum["target"] = df_cusum["close"].shift(-1)
    data = df_cusum.dropna().copy()

    X_train, X_test, y_train, y_test, feature_names = preprocess_split(data.drop(columns=["date"]), "target")
    keep_idx = select_features_mdi(X_train, y_train, feature_names)
    X_train_sel = X_train[:, keep_idx]
    X_test_sel = X_test[:, keep_idx]

    X_train_3d, y_train_3d = make_sliding_window(X_train_sel, y_train, window=1)
    X_test_3d, y_test_3d = make_sliding_window(X_test_sel, y_test, window=1)

    model = build_bilstm((X_train_3d.shape[1], X_train_3d.shape[2]))
    model.fit(X_train_3d, y_train_3d, epochs=10, batch_size=32, verbose=0)
    preds = model.predict(X_test_3d).flatten()

    CUSUM_Control_Chart(preds, y_test_3d, model, X_train_3d, y_train_3d)

In [33]:
# Run for BTC and ETH
run_pipeline("btc", "bitcoin")
# run_pipeline("eth", "ethereum")

Saved: /content/drive/MyDrive/Crypto Research/DATA/Raw/bitcoin_raw.csv
Loaded: /content/drive/MyDrive/Crypto Research/DATA/Raw/bitcoin_raw.csv


/tmp/ipykernel_1529/1465235101.py:21: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  def ROC(series, window): return series.pct_change(periods=window) * 100
/tmp/ipykernel_1529/1465235101.py:21: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  def ROC(series, window): return series.pct_change(periods=window) * 100
/tmp/ipykernel_1529/1465235101.py:21: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  def ROC

Saved: /content/drive/MyDrive/Crypto Research/DATA/Processed/bitcoin_features.csv
Loaded: /content/drive/MyDrive/Crypto Research/DATA/Processed/bitcoin_features.csv
Saved: /content/drive/MyDrive/Crypto Research/DATA/Processed/Intervals/bitcoin/bitcoin_interval1.csv
Skip bitcoin | interval1 | TF 1d (resume)
Skip bitcoin | interval1 | TF 7d (resume)
Skip bitcoin | interval1 | TF 14d (resume)
Skip bitcoin | interval1 | TF 30d (resume)
Skip bitcoin | interval1 | TF 60d (resume)
Skip bitcoin | interval1 | TF 90d (resume)
Saved: /content/drive/MyDrive/Crypto Research/DATA/Processed/Intervals/bitcoin/bitcoin_interval2.csv
Skip bitcoin | interval2 | TF 1d (resume)
Skip bitcoin | interval2 | TF 7d (resume)
Skip bitcoin | interval2 | TF 14d (resume)
Skip bitcoin | interval2 | TF 30d (resume)
Skip bitcoin | interval2 | TF 60d (resume)
Skip bitcoin | interval2 | TF 90d (resume)
Saved: /content/drive/MyDrive/Crypto Research/DATA/Processed/Intervals/bitcoin/bitcoin_interval3.csv
Skip bitcoin | inter

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 56ms/step
CUSUM WARNING: bias detected, retraining on recent data...
